<a href="https://colab.research.google.com/github/stephanie465337/Data-Science-Portfolio-C21/blob/main/NotebookLM/Module%203/4b_BigQuery_Getting_Started.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Documentation and resources

**Big Query**
- Colab has an example notebook on BigQuery too.  File > Open notebook > Examples > Getting Started with BigQuery.
- Also a Big Query Snippets Example Notebook
- [BigQuery Documentation]( https://cloud.google.com/bigquery/docs )
- [Open Data Sets]( https://console.cloud.google.com/marketplace/browse?filter=solution-type:dataset )
- [Reddit - list of data sets available on BQ]( https://www.reddit.com/r/bigquery/wiki/datasets )


**Big Query Console**  
- [Google Cloud Console]( https://console.cloud.google.com )
- Make sure your project is selected
- Scroll down to BigQuery on the left menu
- [Setup and query instructions]( https://cloud.google.com/bigquery/docs/quickstarts/query-public-dataset-console )

**SQL**
- [Kaggle Intro to SQL]( https://www.kaggle.com/learn/intro-to-sql ) uses BigQuery
- [Kaggle Advanced SQL]( https://www.kaggle.com/learn/advanced-sql )

**BigQuery Public Data Sets**
- [Austin Crime]( https://console.cloud.google.com/marketplace/product/city-of-austin/austin-crime )



#  Linking BigQuery to Colab

## Getting started

**You will only need to do this part once.**

1. Use the [Cloud Resource Manager](https://console.cloud.google.com/cloud-resource-manager) to Create a Cloud Platform project if you do not already have one.

    - Create Project
    - Project Name
    - Location

2. [Enable BigQuery APIs](https://console.cloud.google.com/flows/enableapi?apiid=bigquery) for the project

**Note:** You get 1 TB/month of free queries for open datasets
- Kaggle gives you 5 TB/month free



## Imports

In [ ]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
from google.colab import auth
from google.colab import syntax
from google.colab import userdata


In [ ]:
%alias gcloud gcloud

### Provide your credentials

In [ ]:
auth.authenticate_user()
print('Authenticated')

## Optional: Enable data table display

Colab includes the ``google.colab.data_table`` package that can be used to display large pandas dataframes as an interactive data table.
It can be enabled with:

In [ ]:
%load_ext google.colab.data_table
%unload_ext google.colab.data_table

If you would prefer to return to the classic Pandas dataframe display, you can disable this by running:
```python
%unload_ext google.colab.data_table
```

## List projects



In order to query BigQuery, you will need to specify a project ID.  To get a list of project IDs associated with your account, run the following command.

In [ ]:
gcloud projects list --sort-by=projectId

## Declare the Cloud project ID which will be used throughout this notebook

Create a secret in your vault named `bq_billing_project_id` and assign it the value of the project ID you want to use.

1. open the vault ( key icon on the far left )
1. click "Add new secret"
1. type the name ( `bq_billing_project_id` )
1. paste the value ( do NOT type enter )
1. click back in the notebook


Fetch the project ID from your vault.

In [ ]:
project_id = userdata.get('bq_billing_project_id')
project_id

## Samples data set



The [GSOD table](https://console.cloud.google.com/bigquery?p=bigquery-public-data&d=samples&t=gsod&page=table) in the Samples data set contains weather information collected by NOAA, such as precipitation amounts and wind speeds from late 1929 to early 2010.


# Use BigQuery via magics



The `google.cloud.bigquery` library also includes a magic command which runs a query and either displays the result or saves it to a variable as a `DataFrame`.

In [ ]:
# Display query output immediately

%%bigquery --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


In [ ]:
# Save output in a variable `df`

%%bigquery bq_public_data_df --project {project_id}
SELECT
  COUNT(1) as total_rows
FROM `bigquery-public-data.samples.gsod`


In [ ]:
bq_public_data_df

In [ ]:
f'{bq_public_data_df.iloc[0,0]:,.0f}'

# Use BigQuery through google-cloud-bigquery



See [BigQuery documentation](https://cloud.google.com/bigquery/docs) and [library reference documentation](https://googlecloudplatform.github.io/google-cloud-python/latest/bigquery/usage.html).


### Count total number of rows

In [ ]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  bq_df = client.query( query ).to_dataframe()


In [ ]:
bq_df.shape


In [ ]:
bq_df

In [ ]:
query = '''
  SELECT
    COUNT(1) as total
  FROM `bigquery-public-data.samples.gsod`
  '''

with bigquery.Client(project=project_id) as client:
  row_count = client.query( query ).to_dataframe()["total"][0]

print(f'Full dataset has {row_count:,.0f} rows')


## Sample approximately 2000 random rows

### Describe the sampled data

In [ ]:
sample_count = 2000

query = f'''
  SELECT
    *
  FROM
    `bigquery-public-data.samples.gsod`
  WHERE RAND() < {sample_count}/{row_count}
'''

with bigquery.Client(project=project_id) as client:
  df = client.query( query ).to_dataframe()

df


In [ ]:
df.index.size

In [ ]:
df.describe().transpose().astype({"count": int})

### View the first 10 rows

In [ ]:
df.head(10)

In [ ]:
df.isnull().sum()

In [ ]:
# 10 highest total_precipitation samples
(
df
  .sort_values('total_precipitation', ascending=False)
  .head(10)
  [['station_number', 'year', 'month', 'day', 'total_precipitation']]
)


# Use BigQuery through pandas-gbq



The `pandas-gbq` library is a community led project by the pandas community. It covers basic functionality, such as writing a DataFrame to BigQuery and running a query, but as a third-party library it may not handle all BigQuery features or use cases.

[Pandas GBQ Documentation]( https://pandas-gbq.readthedocs.io/en/latest/ )

In [ ]:
query = '''
  SELECT
    name, SUM(number) as count
  FROM
    bigquery-public-data.usa_names.usa_1910_2013
  WHERE
    state = 'TX'
  GROUP BY
    name
  ORDER BY
    count DESC
  LIMIT
    100
'''

df = pandas_gbq.read_gbq( query, project_id=project_id, )
df.head()
